# EDA — estrutura de demanda, variáveis e seleção do modelo OLS

Esta análise exploratória conduz a escolha da estrutura da curva de demanda e o uso das variáveis disponíveis. A sequência é deliberada: começa pelo suporte e pelas distribuições dos dados; testa hipóteses amplas de sazonalidade; define os clusters de nível; compara estruturas de elasticidade; e termina com a sensibilidade do FDS e o diagnóstico dos desvios em relação à curva.

O objetivo não é validar uma especificação fixada de antemão. É documentar, com gráficos e testes, por que o modelo resultante usa preço e dia da semana, como agrupa os dias e onde suas previsões podem ser usadas sem extrapolação.

In [1]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display


ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

ARQ_TREINO = ROOT / "data/interim/.treino.csv"
ARQ_TESTE = ROOT / "data/interim/.teste.csv"
FIGURES_DIR = ROOT / "reports/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

treino = pd.read_csv(ARQ_TREINO, parse_dates=["Data"])
teste = pd.read_csv(ARQ_TESTE, parse_dates=["Data"])

CORES = {
    "Segunda": "#1f77b4",
    "TerQuaQui": "#2ca02c",
    "Sexta": "#ff7f0e",
    "FimDeSemana": "#9467bd",
}
ORDEM_CLUSTER = ["Segunda", "TerQuaQui", "Sexta", "FimDeSemana"]
ROTULO_CLUSTER = {
    "Segunda": "Segunda",
    "TerQuaQui": "Terça + Quarta + Quinta",
    "Sexta": "Sexta",
    "FimDeSemana": "Sábado + Domingo",
}


def definir_cluster(dia):
    if dia == "Segunda":
        return "Segunda"
    if dia in ("Terça", "Quarta", "Quinta"):
        return "TerQuaQui"
    if dia == "Sexta":
        return "Sexta"
    return "FimDeSemana"


def preparar(df, interacoes=False, colunas_modelo=None):
    """Cria matriz log-log; a referência é Terça+Quarta+Quinta."""
    base = df.copy()
    base["Cluster"] = base["Dia da Semana"].map(definir_cluster)
    base["ln_preco"] = np.log(base["Preço"])
    base["ln_volume"] = np.log(base["Volume Realizado (kg)"])

    dummies = pd.get_dummies(base["Cluster"], dtype=float)
    dummies = dummies.drop(columns="TerQuaQui", errors="ignore")
    X = pd.concat([base[["ln_preco"]], dummies], axis=1)
    if interacoes:
        for coluna in dummies.columns:
            X[f"ln_preco_x_{coluna}"] = X["ln_preco"] * X[coluna]
    X = sm.add_constant(X, has_constant="add")
    if colunas_modelo is not None:
        X = X.reindex(columns=colunas_modelo, fill_value=0.0)
    return base, X


def ajustar_ols(df, interacoes=False):
    base, X = preparar(df, interacoes=interacoes)
    return sm.OLS(base["ln_volume"], X).fit(), base, X


def prever_volume(modelo, df, interacoes=False):
    _, X = preparar(df, interacoes=interacoes, colunas_modelo=modelo.params.index)
    return np.exp(modelo.predict(X))


def mape(modelo, df, interacoes=False):
    previsto = prever_volume(modelo, df, interacoes)
    real = df["Volume Realizado (kg)"]
    return (np.abs(real - previsto) / real).mean() * 100


def metricas_previsao(modelo, df, interacoes=False):
    """Métricas no volume original, calculadas sobre a mesma base de previsão."""
    previsto = prever_volume(modelo, df, interacoes).to_numpy()
    real = df["Volume Realizado (kg)"].to_numpy()
    return metricas_de_vetores(real, previsto)


def metricas_de_vetores(real, previsto):
    """Versão genérica para comparar modelos com matrizes de desenho distintas."""
    erro_absoluto = np.abs(real - previsto)
    return {
        "RMSE (kg)": np.sqrt(np.mean((real - previsto) ** 2)),
        "WMAPE volume (%)": erro_absoluto.sum() / real.sum() * 100,
        "MAPE incidência (%)": (erro_absoluto / real).mean() * 100,
        "Viés agregado (%)": (previsto.sum() - real.sum()) / real.sum() * 100,
    }


def elasticidade_por_cluster(modelo, interacoes=False):
    beta = modelo.params["ln_preco"]
    valores = {"TerQuaQui": beta}
    for grupo in ("Segunda", "Sexta", "FimDeSemana"):
        valores[grupo] = beta + (modelo.params.get(f"ln_preco_x_{grupo}", 0.0) if interacoes else 0.0)
    return valores


print(f"Treino: {len(treino)} observações; teste: {len(teste)} observações.")
print("Dias no teste:", ", ".join(teste["Dia da Semana"].unique()))

Treino: 76 observações; teste: 10 observações.
Dias no teste: Segunda, Terça, Quarta, Quinta, Sexta


## 0. Distribuições iniciais e suporte observado

A EDA começa pelas variáveis disponíveis, antes de qualquer agrupamento ou ajuste. Os histogramas mostram o suporte efetivamente observado, a assimetria e a diferença entre média e mediana. Isso orienta a transformação logarítmica de preço e volume usada mais adiante e delimita a faixa em que uma curva estimada pode ser interpretada como interpolação.

Receita é mantida como variável descritiva — ela deriva de preço e volume e, portanto, não entra como explicativa adicional na curva de demanda.

In [2]:
variaveis_iniciais = [
    ("Volume Realizado (kg)", "Volume realizado (kg)", "#2166ac"),
    ("Preço", "Preço realizado (R$/kg)", "#b35806"),
    ("Custo Realizado (R$/kg)", "Custo realizado (R$/kg)", "#542788"),
    ("Receita Realizada (R$)", "Receita realizada (R$)", "#1b7837"),
]

fig, eixos = plt.subplots(2, 2, figsize=(13, 8))
for ax, (coluna, rotulo, cor) in zip(eixos.flat, variaveis_iniciais):
    valores = treino[coluna].dropna()
    ax.hist(valores, bins="fd", color=cor, edgecolor="white", alpha=0.9)
    ax.axvline(valores.mean(), color="#d7301f", lw=2, label=f"Média: {valores.mean():,.2f}")
    ax.axvline(valores.median(), color="#252525", lw=2, ls="--", label=f"Mediana: {valores.median():,.2f}")
    ax.set(title=f"Distribuição de {rotulo}", xlabel=rotulo, ylabel="Número de dias")
    ax.grid(axis="y", alpha=0.2)
    ax.legend(fontsize=8)
fig.suptitle("Variáveis iniciais — dados observados no treino", y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_ols_histogramas_iniciais.png", dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)

suporte_observado = pd.DataFrame([
    {
        "Variável": rotulo,
        "Mínimo": treino[coluna].min(),
        "Mediana": treino[coluna].median(),
        "Máximo": treino[coluna].max(),
    }
    for coluna, rotulo, _ in variaveis_iniciais
])
display(suporte_observado.round(2))

<Figure size 1300x800 with 4 Axes>

,Variável,Mínimo,Mediana,Máximo
0,Volume realizado (kg),0.59,16.70,100.00
1,Preço realizado (R$/kg),1188.86,1276.49,1365.49
2,Custo realizado (R$/kg),949.12,983.67,1030.34
3,Receita realizada (R$),699.06,21799.94,120163.17


**Leitura inicial.** Preço apresenta uma faixa observada relativamente concentrada, enquanto volume e receita são mais assimétricos. Essa assimetria é compatível com a formulação multiplicativa da demanda e motiva trabalhar com `ln(volume)` e `ln(preço)`. Os histogramas descrevem cada variável isoladamente: não identificam sazonalidade, causalidade nem pontos “fora da curva”; essas perguntas são avaliadas nas etapas seguintes.

## 1. Triagem de sazonalidade: mês, quinzena, semana do mês ou dia da semana?

Antes de formar clusters, consideramos quatro hipóteses de calendário sem pressupor que o efeito venha do dia da semana. Para cada uma, os gráficos mostram a distribuição e a média de **volume** e de **preço**; em seguida, o teste OLS avalia o efeito de calendário sobre o volume **controlando por preço**. Assim, uma diferença de volume que seja apenas consequência de preços distintos não é interpretada como sazonalidade de nível.

Os gráficos desta etapa são descritivos — não são previsões nem extrapolações.

In [3]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

calendario = treino.copy()
calendario["Mês"] = calendario["Data"].dt.month.map({8: "Ago", 9: "Set", 10: "Out"})
calendario["Quinzena"] = np.where(calendario["Data"].dt.day <= 15, "1ª quinzena", "2ª quinzena")
calendario["Semana do mês"] = "Semana " + (((calendario["Data"].dt.day - 1) // 7) + 1).astype(str)


def plotar_perfil_calendario(dados, coluna, ordem, titulo, arquivo):
    """Perfil descritivo; nenhuma curva ou valor estimado é mostrado."""
    base = dados.copy()
    base[coluna] = pd.Categorical(base[coluna], categories=ordem, ordered=True)
    grupos_volume = [base.loc[base[coluna] == grupo, "Volume Realizado (kg)"].dropna() for grupo in ordem]
    grupos_preco = [base.loc[base[coluna] == grupo, "Preço"].dropna() for grupo in ordem]
    media_volume = base.groupby(coluna, observed=False)["Volume Realizado (kg)"].mean().reindex(ordem)
    media_preco = base.groupby(coluna, observed=False)["Preço"].mean().reindex(ordem)

    fig, eixos = plt.subplots(2, 2, figsize=(13, 8))
    estilo_caixa_volume = {"facecolor": "#c6dbef", "edgecolor": "#2171b5"}
    estilo_caixa_preco = {"facecolor": "#fdd0a2", "edgecolor": "#d94801"}
    comum = {
        "patch_artist": True,
        "showmeans": True,
        "medianprops": {"color": "#252525", "linewidth": 2},
        "meanprops": {"marker": "D", "markerfacecolor": "#d7301f", "markeredgecolor": "#d7301f", "markersize": 5},
        "whiskerprops": {"color": "#525252"},
        "capprops": {"color": "#525252"},
        "flierprops": {"marker": "o", "markerfacecolor": "none", "markeredgecolor": "#f16913", "markersize": 5},
    }
    eixos[0, 0].boxplot(grupos_volume, tick_labels=ordem, boxprops=estilo_caixa_volume, **comum)
    eixos[0, 0].set(title=f"Distribuição de volume por {coluna.lower()}", ylabel="Volume diário (kg)")
    eixos[0, 1].bar(ordem, media_volume, color="#2166ac")
    eixos[0, 1].set(title=f"Volume médio diário por {coluna.lower()}", ylabel="Média de volume (kg)")
    eixos[1, 0].boxplot(grupos_preco, tick_labels=ordem, boxprops=estilo_caixa_preco, **comum)
    eixos[1, 0].set(title=f"Distribuição de preço por {coluna.lower()}", xlabel=coluna, ylabel="Preço (R$/kg)")
    eixos[1, 1].bar(ordem, media_preco, color="#b35806")
    eixos[1, 1].set(title=f"Preço médio por {coluna.lower()}", xlabel=coluna, ylabel="Média de preço (R$/kg)")
    for ax in eixos.flat:
        ax.grid(axis="y", alpha=0.2)
    fig.legend(
        handles=[
            Patch(facecolor="#c6dbef", edgecolor="#2171b5", label="Caixa azul: Q1–Q3 do volume"),
            Patch(facecolor="#fdd0a2", edgecolor="#d94801", label="Caixa laranja: Q1–Q3 do preço"),
            Line2D([0], [0], color="#252525", lw=2, label="Linha: mediana"),
            Line2D([0], [0], marker="D", color="none", markerfacecolor="#d7301f", label="Losango: média"),
            Line2D([0], [0], marker="o", color="none", markerfacecolor="none", markeredgecolor="#f16913", label="Círculo: além de 1,5 × IQR"),
        ],
        loc="upper center", ncol=3, frameon=True, bbox_to_anchor=(0.5, 0.96),
    )
    fig.suptitle(titulo, y=1.02, fontsize=14)
    fig.tight_layout(rect=(0, 0, 1, 0.88))
    fig.savefig(FIGURES_DIR / arquivo, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)


plotar_perfil_calendario(calendario, "Mês", ["Ago", "Set", "Out"], "Sazonalidade mensal — descrição dos dados observados", "eda_ols_sazonalidade_mes.png")
plotar_perfil_calendario(calendario, "Quinzena", ["1ª quinzena", "2ª quinzena"], "Sazonalidade por quinzena — descrição dos dados observados", "eda_ols_sazonalidade_quinzena.png")
plotar_perfil_calendario(calendario, "Semana do mês", ["Semana 1", "Semana 2", "Semana 3", "Semana 4", "Semana 5"], "Sazonalidade por semana do mês — descrição dos dados observados", "eda_ols_sazonalidade_semana_mes.png")
plotar_perfil_calendario(calendario, "Dia da Semana", ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"], "Sazonalidade por dia da semana — descrição dos dados observados", "eda_ols_sazonalidade_dia_semana.png")


def testar_sazonalidade(df, coluna):
    base = df.copy()
    base["ln_preco"] = np.log(base["Preço"])
    base["ln_volume"] = np.log(base["Volume Realizado (kg)"])
    dummies = pd.get_dummies(base[coluna], dtype=float)
    referencia = dummies.sum().idxmax()
    dummies = dummies.drop(columns=referencia)
    modelo_sem_calendario = sm.OLS(base["ln_volume"], sm.add_constant(base[["ln_preco"]], has_constant="add")).fit()
    modelo_com_calendario = sm.OLS(
        base["ln_volume"], sm.add_constant(pd.concat([base[["ln_preco"]], dummies], axis=1), has_constant="add")
    ).fit()
    f_estat, p_valor, gl = modelo_com_calendario.compare_f_test(modelo_sem_calendario)
    return {
        "Hipótese de calendário": coluna,
        "Grupos": len(dummies.columns) + 1,
        "Referência": referencia,
        "F conjunto": f_estat,
        "p-valor": p_valor,
        "BIC sem calendário": modelo_sem_calendario.bic,
        "BIC com calendário": modelo_com_calendario.bic,
    }


resultado_sazonalidade = pd.DataFrame([
    testar_sazonalidade(calendario, "Mês"),
    testar_sazonalidade(calendario, "Quinzena"),
    testar_sazonalidade(calendario, "Semana do mês"),
    testar_sazonalidade(calendario, "Dia da Semana"),
])
display(resultado_sazonalidade.round(3))

<Figure size 1300x800 with 4 Axes>

<Figure size 1300x800 with 4 Axes>

<Figure size 1300x800 with 4 Axes>

<Figure size 1300x800 with 4 Axes>

,Hipótese de calendário,Grupos,Referência,F conjunto,p-valor,BIC sem calendário,BIC com calendário
0,Mês,3,Out,0.361,0.698,246.5,254.403
1,Quinzena,2,2ª quinzena,0.311,0.579,246.5,250.508
2,Semana do mês,5,Semana 4,0.337,0.852,246.5,262.373
3,Dia da Semana,7,Sexta,52.798,0.000,246.5,140.763


**Conclusão da triagem.** Depois de controlar por preço, mês (p = 0,698), quinzena (p = 0,579) e semana do mês (p = 0,852) não apresentam evidência de efeito de nível; todos ainda pioram o BIC. O dia da semana, em contraste, é fortemente associado ao volume (p < 0,001) e reduz o BIC de forma expressiva. Por isso, apenas o dia da semana segue para a etapa de agrupamento; mês, quinzena e semana do mês não entram como variáveis do modelo.

## 2. É possível agrupar (cluster) ?

O agrupamento não é uma premissa colocada diretamente no modelo. A sequência
abaixo começa com os sete dias separados, controla o efeito de preço e só
então avalia quais diferenças de **nível** podem ser removidas sem perda
preditiva ou de parcimônia. A pergunta específica é: depois de controlar por
preço, Terça e Quinta têm nível de demanda distinguível de Quarta?

In [4]:
ORDEM_DIAS = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]

fig, ax = plt.subplots(figsize=(10, 5.3))
dados_boxplot = [treino.loc[treino["Dia da Semana"] == dia, "Volume Realizado (kg)"].to_numpy() for dia in ORDEM_DIAS]
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ax.boxplot(
    dados_boxplot,
    tick_labels=ORDEM_DIAS,
    patch_artist=True,
    showmeans=True,
    boxprops={"facecolor": "#c6dbef", "edgecolor": "#2171b5"},
    medianprops={"color": "#08519c", "linewidth": 2},
    meanprops={"marker": "D", "markerfacecolor": "#d7301f", "markeredgecolor": "#d7301f", "markersize": 5},
    whiskerprops={"color": "#525252"},
    capprops={"color": "#525252"},
    flierprops={"marker": "o", "markerfacecolor": "none", "markeredgecolor": "#f16913", "markersize": 5},
)
ax.set(title="Ponto de partida: distribuição de volume por dia", xlabel="Dia da semana", ylabel="Volume realizado (kg)")
ax.grid(axis="y", alpha=0.2)
ax.legend(
    handles=[
        Patch(facecolor="#c6dbef", edgecolor="#2171b5", label="Caixa: 1º a 3º quartil (Q1–Q3)"),
        Line2D([0], [0], color="#08519c", lw=2, label="Linha: mediana"),
        Line2D([0], [0], marker="D", color="none", markerfacecolor="#d7301f", markeredgecolor="#d7301f", label="Losango: média"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="none", markeredgecolor="#f16913", label="Círculo: além de 1,5 × IQR"),
    ],
    loc="upper right",
    frameon=True,
)
fig.tight_layout()
display(fig)
plt.close(fig)


def ajustar_particao(df, mapa_dias, referencia=None):
    """OLS log-log com uma elasticidade e interceptos definidos por `mapa_dias`."""
    base = df.copy()
    base["Grupo"] = base["Dia da Semana"].map(mapa_dias)
    base["ln_preco"] = np.log(base["Preço"])
    base["ln_volume"] = np.log(base["Volume Realizado (kg)"])
    dummies = pd.get_dummies(base["Grupo"], dtype=float)
    referencia = referencia or dummies.sum().idxmax()
    dummies = dummies.drop(columns=referencia, errors="ignore")
    X = sm.add_constant(pd.concat([base[["ln_preco"]], dummies], axis=1), has_constant="add")
    return sm.OLS(base["ln_volume"], X).fit(), referencia


def prever_particao(modelo, df, mapa_dias, referencia):
    base = df.copy()
    base["Grupo"] = base["Dia da Semana"].map(mapa_dias)
    base["ln_preco"] = np.log(base["Preço"])
    dummies = pd.get_dummies(base["Grupo"], dtype=float).drop(columns=referencia, errors="ignore")
    X = sm.add_constant(pd.concat([base[["ln_preco"]], dummies], axis=1), has_constant="add")
    X = X.reindex(columns=modelo.params.index, fill_value=0.0)
    return np.exp(modelo.predict(X))


mapa_dia_a_dia = {dia: dia for dia in ORDEM_DIAS}
modelo_dia_a_dia, _ = ajustar_particao(treino, mapa_dia_a_dia, referencia="Quarta")
teste_conjunto_tqq = modelo_dia_a_dia.f_test("Terça = 0, Quinta = 0")

tabela_niveis = pd.DataFrame({
    "Dia vs. Quarta": ["Terça", "Quinta"],
    "Diferença de intercepto": [modelo_dia_a_dia.params["Terça"], modelo_dia_a_dia.params["Quinta"]],
    "p-valor individual": [modelo_dia_a_dia.pvalues["Terça"], modelo_dia_a_dia.pvalues["Quinta"]],
})
display(tabela_niveis.round(3))
print(f"Teste F conjunto (Terça = Quarta e Quinta = Quarta): F={float(teste_conjunto_tqq.fvalue):.3f}; p={float(teste_conjunto_tqq.pvalue):.3f}")

<Figure size 1000x530 with 1 Axes>

,Dia vs. Quarta,Diferença de intercepto,p-valor individual
0,Terça,-0.038,0.853
1,Quinta,-0.051,0.800


Teste F conjunto (Terça = Quarta e Quinta = Quarta): F=0.035; p=0.966


**Como ler o boxplot.** A caixa azul mostra os 50% centrais dos volumes (de Q1 a Q3); a linha azul escura é a mediana e o losango vermelho é a média. As hastes alcançam os valores até 1,5 vezes o intervalo interquartil (IQR), e círculos laranja ficam além desse limite. Esses círculos são apenas um resumo da distribuição marginal de volume por dia: eles **não** classificam nem removem “outliers” da curva de demanda. Essa classificação é feita depois, condicionalmente a preço e cluster, pelo gráfico de resíduos da etapa 6.

Terça e Quinta não apresentam diferença estatisticamente identificável em
relação a Quarta depois de controlar por preço; o teste conjunto também não
rejeita as duas igualdades. Isso não “prova igualdade”, sobretudo com amostra
curta, mas fornece evidência para testar uma partição mais simples. A decisão
final abaixo exige ainda que separar esses dias não traga ganho convincente de
BIC ou de erro fora da amostra.

In [5]:
mapa_final = {dia: definir_cluster(dia) for dia in ORDEM_DIAS}
mapa_segunda_absorvida = {
    dia: "SegTerQuaQui" if dia in ("Segunda", "Terça", "Quarta", "Quinta") else "Sexta" if dia == "Sexta" else "FimDeSemana"
    for dia in ORDEM_DIAS
}
mapa_quinta_separada = {
    dia: "Segunda" if dia == "Segunda" else "TerQua" if dia in ("Terça", "Quarta") else "Quinta" if dia == "Quinta" else "Sexta" if dia == "Sexta" else "FimDeSemana"
    for dia in ORDEM_DIAS
}
mapa_terca_separada = {
    dia: "Segunda" if dia == "Segunda" else "Terça" if dia == "Terça" else "QuaQui" if dia in ("Quarta", "Quinta") else "Sexta" if dia == "Sexta" else "FimDeSemana"
    for dia in ORDEM_DIAS
}

particoes = [
    ("Dias separados", mapa_dia_a_dia),
    ("Final: Seg / TerQuaQui / Sex / FDS", mapa_final),
    ("Segunda absorvida em TerQuaQui", mapa_segunda_absorvida),
    ("Quinta separada de Ter+Qua", mapa_quinta_separada),
    ("Terça separada de Qua+Qui", mapa_terca_separada),
]
linhas_particao = []
for nome, mapa in particoes:
    modelo, referencia = ajustar_particao(treino, mapa)
    previsto = prever_particao(modelo, teste, mapa, referencia)
    real = teste["Volume Realizado (kg)"].to_numpy()
    linhas_particao.append({
        "Partição": nome,
        "Parâmetros": len(modelo.params),
        "BIC": modelo.bic,
        "MAPE teste (%)": (np.abs(real - previsto) / real).mean() * 100,
    })
display(pd.DataFrame(linhas_particao).round(2))

,Partição,Parâmetros,BIC,MAPE teste (%)
0,Dias separados,8,140.76,55.64
1,Final: Seg / TerQuaQui / Sex / FDS,5,130.78,56.07
2,Segunda absorvida em TerQuaQui,4,130.85,58.61
3,Quinta separada de Ter+Qua,6,135.07,56.28
4,Terça separada de Qua+Qui,6,135.10,55.86


A partição final é preferida porque separar Terça ou Quinta individualmente
aumenta o BIC sem ganho relevante de MAPE. O modelo dia a dia reduz o MAPE em
um único teste de apenas dez dias, mas cobra três parâmetros extras e tem BIC
substancialmente pior; é evidência de flexibilidade, não de generalização
comprovada. A comparação “Segunda absorvida” permanece como sensibilidade de
fronteira; o FDS é testado separadamente contra Sexta na próxima etapa.

## 3. Estrutura de nível: os quatro grupos usados no modelo

O agrupamento avaliado é: Segunda; Terça+Quarta+Quinta; Sexta; e FDS. O FDS
permanece por duas razões distintas: recebe um intercepto próprio e é a única
fonte de informação para esse nível de demanda. Isso não implica estimar uma
elasticidade exclusiva para apenas dez observações.

In [6]:
treino_plot = treino.assign(Cluster=treino["Dia da Semana"].map(definir_cluster))
resumo = (
    treino_plot.groupby("Cluster", observed=True)
    .agg(n=("Data", "size"), preco_mediano=("Preço", "median"), volume_mediano=("Volume Realizado (kg)", "median"))
    .reindex(ORDEM_CLUSTER)
)
display(resumo.rename(index=ROTULO_CLUSTER).round(2))

fig, eixos = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)
for cluster in ORDEM_CLUSTER:
    dados = treino_plot[treino_plot["Cluster"] == cluster]
    rotulo = f"{ROTULO_CLUSTER[cluster]} (n={len(dados)})"
    eixos[0].scatter(dados["Preço"], dados["Volume Realizado (kg)"], s=58, alpha=0.85,
                     color=CORES[cluster], label=rotulo)
    dados_ordenados = dados.sort_values("Preço")
    eixos[1].plot(dados_ordenados["Preço"], dados_ordenados["Volume Realizado (kg)"], marker="o", markersize=5,
                  lw=1.5, alpha=0.75, color=CORES[cluster], label=rotulo)

eixos[0].set(title="Dispersão observada: preço × volume", xlabel="Preço realizado (R$)", ylabel="Volume realizado (kg)")
eixos[1].set(title="Mesmos pontos unidos por preço crescente", xlabel="Preço realizado (R$)")
for ax in eixos:
    ax.grid(alpha=0.2)
eixos[0].legend(title="Cluster", frameon=True)
fig.suptitle("Preço e volume observados por cluster", y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_ols_dispersao_preco_volume.png", dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)

,n,preco_mediano,volume_mediano
Cluster,,,
Segunda,13,1282.40,31.31
Terça + Quarta + Quinta,39,1277.43,21.66
Sexta,14,1265.12,6.83
Sábado + Domingo,10,1267.29,1.67


<Figure size 1600x550 with 2 Axes>

**Como ler os painéis.** O painel da esquerda mantém a dispersão sem ordem. No painel da direita, os mesmos pontos são conectados por **preço crescente dentro de cada cluster**, apenas para tornar visível a trajetória observada. Essa linha não é uma curva de demanda estimada, não representa a ordem cronológica dos dias e não deve ser extrapolada; a comparação entre curvas OLS vem na etapa 4.

### Teste de nível: FDS merece intercepto próprio?

Esta é uma pergunta diferente da elasticidade. Comparamos dois modelos OLS
aninhados, ambos com inclinação de preço compartilhada: o modelo completo tem
quatro interceptos; o reduzido força FDS e Sexta a terem o mesmo nível. O
teste F avalia se o intercepto adicional de FDS explica sinal além do acaso;
BIC e MAPE de teste são apresentados como critérios complementares.

In [7]:
def ajustar_fds_absorvido(df):
    """Modelo reduzido: FDS compartilha o intercepto da Sexta."""
    base = df.copy()
    base["Cluster_reduzido"] = base["Dia da Semana"].map(definir_cluster).replace({"FimDeSemana": "Sexta"})
    base["ln_preco"] = np.log(base["Preço"])
    base["ln_volume"] = np.log(base["Volume Realizado (kg)"])
    dummies = pd.get_dummies(base["Cluster_reduzido"], dtype=float).drop(columns="TerQuaQui", errors="ignore")
    X = sm.add_constant(pd.concat([base[["ln_preco"]], dummies], axis=1), has_constant="add")
    return sm.OLS(base["ln_volume"], X).fit(), base


def prever_fds_absorvido(modelo, df):
    base = df.copy()
    base["Cluster_reduzido"] = base["Dia da Semana"].map(definir_cluster).replace({"FimDeSemana": "Sexta"})
    base["ln_preco"] = np.log(base["Preço"])
    dummies = pd.get_dummies(base["Cluster_reduzido"], dtype=float).drop(columns="TerQuaQui", errors="ignore")
    X = sm.add_constant(pd.concat([base[["ln_preco"]], dummies], axis=1), has_constant="add")
    X = X.reindex(columns=modelo.params.index, fill_value=0.0)
    return np.exp(modelo.predict(X))


modelo_quatro_niveis, _, _ = ajustar_ols(treino)
modelo_fds_sexta, _ = ajustar_fds_absorvido(treino)
f_estat, p_valor, graus_liberdade = modelo_quatro_niveis.compare_f_test(modelo_fds_sexta)
metricas_quatro_niveis = metricas_previsao(modelo_quatro_niveis, teste)
metricas_reduzido = metricas_de_vetores(
    teste["Volume Realizado (kg)"].to_numpy(), prever_fds_absorvido(modelo_fds_sexta, teste).to_numpy()
)

teste_nivel = pd.DataFrame([
    {"Modelo": "4 interceptos: FDS próprio", "Parâmetros": len(modelo_quatro_niveis.params), "BIC": modelo_quatro_niveis.bic, **metricas_quatro_niveis},
    {"Modelo": "3 interceptos: FDS junto da Sexta", "Parâmetros": len(modelo_fds_sexta.params), "BIC": modelo_fds_sexta.bic, **metricas_reduzido},
])
display(teste_nivel.round(2))
print(f"Teste F aninhado para o intercepto de FDS: F={f_estat:.2f}; p={p_valor:.4g}; gl adicional={int(graus_liberdade)}")

,Modelo,Parâmetros,BIC,RMSE (kg),WMAPE volume (%),MAPE incidência (%),Viés agregado (%)
0,4 interceptos: FDS próprio,5,130.78,8.89,38.33,56.07,-25.95
1,3 interceptos: FDS junto da Sexta,4,172.18,8.84,37.69,42.68,-27.09


Teste F aninhado para o intercepto de FDS: F=58.59; p=7.309e-11; gl adicional=1


O teste rejeita juntar FDS à Sexta, e o BIC também favorece o quarto
intercepto. Assim, a EDA mantém FDS como **cluster de nível**. Isso não é
evidência de que FDS tenha elasticidade própria; essa hipótese é avaliada
separadamente abaixo.

## 4. Curvas de demanda OLS: elasticidade única vs. elasticidade por cluster

Ambos os modelos mantêm os quatro interceptos. A única diferença é que o
modelo à direita adiciona três interações `ln(preço) × cluster`, permitindo
inclinações distintas. Assim, a comparação isola exatamente a decisão sobre
elasticidade — não mistura elasticidade com diferenças de nível.

Os dois painéis usam escala log-log: uma curva de potência aparece como uma reta, cuja inclinação é a elasticidade. Cada segmento estimado é desenhado apenas entre o menor e o maior preço observado no respectivo cluster. Não há prolongamento das curvas fora desse suporte; qualquer preço fora desse intervalo seria extrapolação e não é avaliado neste gráfico.

In [8]:
modelo_unico, base_unico, _ = ajustar_ols(treino, interacoes=False)
modelo_cluster, base_cluster, _ = ajustar_ols(treino, interacoes=True)

elast_unica = elasticidade_por_cluster(modelo_unico)
elast_cluster = elasticidade_por_cluster(modelo_cluster, interacoes=True)
display(pd.DataFrame({"Única": elast_unica, "Por cluster": elast_cluster}).rename(index=ROTULO_CLUSTER).round(3))

CORES_CURVAS = {
    "Segunda": "#1f77b4",
    "TerQuaQui": "#ff7f0e",
    "Sexta": "#2ca02c",
    "FimDeSemana": "#d62728",
}
fig, eixos = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
for ax, modelo, interacoes, titulo in [
    (eixos[0], modelo_unico, False, f"Elasticidade única (β={modelo_unico.params['ln_preco']:.2f})"),
    (eixos[1], modelo_cluster, True, "Elasticidade por cluster"),
]:
    elasticidades = elasticidade_por_cluster(modelo, interacoes=interacoes)
    for cluster in ORDEM_CLUSTER:
        obs = treino_plot[treino_plot["Cluster"] == cluster]
        cor = CORES_CURVAS[cluster]
        ax.scatter(obs["Preço"], obs["Volume Realizado (kg)"], color=cor, s=42, alpha=0.65)
        # A curva é avaliada somente no intervalo de preço já observado nesse cluster.
        faixa_preco_cluster = np.linspace(obs["Preço"].min(), obs["Preço"].max(), 200)
        grade = pd.DataFrame({"Preço": faixa_preco_cluster, "Volume Realizado (kg)": 1.0, "Dia da Semana": "Terça"})
        dia_representativo = {"Segunda": "Segunda", "TerQuaQui": "Terça", "Sexta": "Sexta", "FimDeSemana": "Sábado"}[cluster]
        grade["Dia da Semana"] = dia_representativo
        rotulo = ROTULO_CLUSTER[cluster] if not interacoes else f"{ROTULO_CLUSTER[cluster]}: β={elasticidades[cluster]:.2f}"
        ax.plot(faixa_preco_cluster, prever_volume(modelo, grade, interacoes), color=cor, lw=2.5, label=rotulo)
    ax.set(title=titulo, xlabel="Preço observado (R$/kg) — escala log")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.grid(alpha=0.35, which="both", ls="--")
eixos[0].set_ylabel("Volume observado / previsto (kg) — escala log")
eixos[0].legend(title="Cluster", frameon=True)
eixos[1].legend(title="Elasticidade estimada", frameon=True)
fig.suptitle("Curvas de demanda OLS: elasticidade única vs. elasticidade por cluster", y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_ols_curvas_unica_vs_cluster.png", dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)

,Única,Por cluster
Terça + Quarta + Quinta,-12.458,-15.908
Segunda,-12.458,-13.481
Sexta,-12.458,-13.558
Sábado + Domingo,-12.458,1.113


<Figure size 1600x600 with 2 Axes>

### 4.1. Métricas de previsão e decisão da elasticidade

Após estimar e visualizar as duas estruturas de elasticidade, as métricas são calculadas no volume original (kg) e sempre no mesmo teste de dez dias. Elas não são uma validação posterior de um modelo já escolhido: são parte da comparação que decide entre as duas estruturas. **RMSE** penaliza proporcionalmente mais os erros grandes em kg,
sendo útil quando uma falha em alto volume é mais grave para planejamento.

**WMAPE por volume** é a definição padrão: `Σ|volume real − previsto| / Σvolume real`.
Ela equivale a uma média dos erros percentuais ponderada pelo volume real e,
portanto, dá mais peso aos dias de maior venda. Já o **MAPE por incidência**
dá o mesmo peso a cada dia. Neste dado há uma observação por dia; se
“incidência” significar frequência de linhas, seus pesos são todos 1 e a
métrica é exatamente o MAPE, não outro WMAPE.

O **viés agregado** é `(Σvolume previsto − Σvolume real) / Σvolume real`.
Ele mede direção, não magnitude: valor positivo indica superestimação do
volume total e negativo, subestimação. Para uma meta de volume no horizonte
de precificação, convém que seu valor fique próximo de zero, mas ele deve ser
lido junto de RMSE e WMAPE — vieses opostos podem se cancelar e mascarar um
erro absoluto alto.

Nenhuma das duas deve ser renomeada como “WMAPE por incidência”. Uma eventual
ponderação por prioridade operacional (por exemplo, dias do horizonte de
otimização) precisa declarar seus pesos e ser reportada como métrica própria.

In [9]:
metricas_unico = metricas_previsao(modelo_unico, teste)
metricas_cluster = metricas_previsao(modelo_cluster, teste, interacoes=True)
comparacao = pd.DataFrame(
    [
        {
            "Modelo": "Elasticidade única",
            "Parâmetros": len(modelo_unico.params),
            "R² ajustado": modelo_unico.rsquared_adj,
            "BIC": modelo_unico.bic,
            **metricas_unico,
        },
        {
            "Modelo": "Elasticidade por cluster",
            "Parâmetros": len(modelo_cluster.params),
            "R² ajustado": modelo_cluster.rsquared_adj,
            "BIC": modelo_cluster.bic,
            **metricas_cluster,
        },
    ]
)
display(comparacao.round(2))

,Modelo,Parâmetros,R² ajustado,BIC,RMSE (kg),WMAPE volume (%),MAPE incidência (%),Viés agregado (%)
0,Elasticidade única,5,0.82,130.78,8.89,38.33,56.07,-25.95
1,Elasticidade por cluster,8,0.84,133.04,9.64,42.67,57.90,-31.46


**Decisão da elasticidade.** No painel da direita, as inclinações possíveis ficam visíveis — inclusive a inclinação positiva de FDS, sinal instável estimado com apenas dez observações. A tabela mostra que o modelo por cluster ganha ajuste dentro da amostra, como esperado por ter mais parâmetros, mas tem BIC maior e piora RMSE, WMAPE por volume e MAPE por incidência no teste. Portanto, a escolha é a **elasticidade única compartilhada**; as elasticidades por cluster ficam como diagnóstico visual da incerteza, não como especificação principal.

## 5. Sensibilidade obrigatória: incluir ou retirar o FDS

A comparação abaixo não pergunta se FDS tem nível próprio — ele tem. Pergunta
quanto as dez observações de FDS alteram o parâmetro global de preço. Como o
teste não contém FDS, a comparação de MAPE avalia somente dias úteis nos dois
ajustes. BIC não é comparável aqui, pois o tamanho amostral muda de 76 para 66.

In [10]:
dias_uteis = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta"]
treino_uteis = treino[treino["Dia da Semana"].isin(dias_uteis)].copy()
modelo_uteis, _, _ = ajustar_ols(treino_uteis, interacoes=False)
metricas_uteis = metricas_previsao(modelo_uteis, teste)

sensibilidade_fds = pd.DataFrame(
    [
        {
            "Ajuste": "Principal: 4 interceptos, com FDS",
            "N treino": len(treino),
            "Elasticidade OLS": modelo_unico.params["ln_preco"],
            **metricas_unico,
        },
        {
            "Ajuste": "Sensibilidade: somente dias úteis",
            "N treino": len(treino_uteis),
            "Elasticidade OLS": modelo_uteis.params["ln_preco"],
            **metricas_uteis,
        },
    ]
)
sensibilidade_fds["Variação vs. principal (%)"] = (
    (sensibilidade_fds["Elasticidade OLS"] / modelo_unico.params["ln_preco"] - 1) * 100
)
display(sensibilidade_fds.round(2))

fig, ax = plt.subplots(figsize=(8, 4.5))
cores = ["#1f77b4", "#d62728"]
barras = ax.bar(sensibilidade_fds["Ajuste"], sensibilidade_fds["Elasticidade OLS"], color=cores, width=0.6)
for barra, valor in zip(barras, sensibilidade_fds["Elasticidade OLS"]):
    ax.text(barra.get_x() + barra.get_width() / 2, valor - 0.35, f"{valor:.2f}", ha="center", va="top", color="white", fontweight="bold")
ax.axhline(modelo_unico.params["ln_preco"], color="#1f77b4", lw=1, ls="--")
ax.set(title="Sensibilidade da elasticidade à presença do FDS", ylabel="Elasticidade OLS (β)")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_ols_sensibilidade_fds.png", dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)

,Ajuste,N treino,Elasticidade OLS,RMSE (kg),WMAPE volume (%),MAPE incidência (%),Viés agregado (%),Variação vs. principal (%)
0,"Principal: 4 interceptos, com FDS",76,-12.46,8.89,38.33,56.07,-25.95,0.00
1,Sensibilidade: somente dias úteis,66,-14.94,9.66,42.79,56.49,-31.53,19.95


<Figure size 800x450 with 1 Axes>

O resultado é material para interpretação: a elasticidade passa de cerca de
**−12,46** para **−14,94** ao retirar o FDS. O MAPE por incidência quase não
muda (56,07% → 56,49%), mas as métricas sensíveis a volume pioram: RMSE de
8,89 para 9,66 kg e WMAPE de 38,33% para 42,79%. Assim, manter FDS no ajuste
principal é favorecido também por erro preditivo agregado; a contribuição do
FDS para a inclinação global continua incerta e deve ser reportada como
análise de sensibilidade.

## 6. Pontos que divergem da curva: diagnóstico por resíduo, sem remoção

Nesta EDA, “ponto fora da curva” significa erro de previsão no espaço
logarítmico depois de controlar simultaneamente por preço e cluster. Não é um
valor extremo de preço ou volume analisado isoladamente. O gráfico serve para
localizar esses desvios e discutir contexto operacional; não exclui nem altera
observações.

In [11]:
diagnostico = base_unico[["Data", "Dia da Semana", "Cluster", "Preço", "Volume Realizado (kg)", "ln_volume"]].copy()
diagnostico["ln_volume_previsto"] = modelo_unico.fittedvalues
diagnostico["residuo_log"] = modelo_unico.resid
diagnostico["desvio_percentual"] = (np.exp(diagnostico["residuo_log"]) - 1) * 100

limite_visual = 2 * diagnostico["residuo_log"].std(ddof=int(modelo_unico.df_model) + 1)
diagnostico["destacado"] = diagnostico["residuo_log"].abs() > limite_visual
destacados = diagnostico.loc[diagnostico["destacado"], ["Data", "Dia da Semana", "Preço", "Volume Realizado (kg)", "desvio_percentual"]].copy()
destacados[["Preço", "Volume Realizado (kg)", "desvio_percentual"]] = destacados[["Preço", "Volume Realizado (kg)", "desvio_percentual"]].round(2)
display(destacados)

fig, ax = plt.subplots(figsize=(10, 5.3))
for cluster in ORDEM_CLUSTER:
    dados = diagnostico[diagnostico["Cluster"] == cluster]
    ax.scatter(dados["ln_volume_previsto"], dados["residuo_log"], color=CORES[cluster], s=58, alpha=0.85,
               label=ROTULO_CLUSTER[cluster])
for _, ponto in diagnostico[diagnostico["destacado"]].iterrows():
    ax.annotate(ponto["Data"].strftime("%d/%m"), (ponto["ln_volume_previsto"], ponto["residuo_log"]),
                xytext=(5, 5), textcoords="offset points", fontsize=8)
ax.axhline(0, color="black", lw=1)
ax.axhline(limite_visual, color="#d62728", lw=1, ls="--", label="limite visual ±2σ")
ax.axhline(-limite_visual, color="#d62728", lw=1, ls="--")
ax.set(title="Diagnóstico correto: resíduo em torno da curva OLS", xlabel="ln(volume previsto)", ylabel="Resíduo em ln(volume)")
ax.grid(alpha=0.2)
ax.legend(ncol=2, frameon=True)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_ols_residuos_curva.png", dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)

,Data,Dia da Semana,Preço,Volume Realizado (kg),desvio_percentual
22,2025-08-30,Sábado,1294.79,3.68,262.59
57,2025-10-11,Sábado,1188.86,0.59,-80.01
64,2025-10-19,Domingo,1233.57,0.59,-68.34
68,2025-10-23,Quinta,1344.63,2.82,-75.09


<Figure size 1000x530 with 1 Axes>

**Leitura do resultado.** Os pontos destacados são os que ficam além de ±2 desvios-padrão dos resíduos logarítmicos: seu volume observado foi muito maior ou menor que o previsto **dado simultaneamente o preço e o cluster**. No ajuste atual, 30/08 (sábado) está acima da curva; 11/10 (sábado), 19/10 (domingo) e 23/10 (quinta) estão abaixo. São sinais para investigar contexto operacional, não valores a excluir: todos permanecem no ajuste OLS. O limiar é apenas visual e não substitui uma decisão de negócio ou uma análise de influência.

## 7. Governança da evidência: o que o teste atual ainda pode dizer?

Os 10 dias finais foram consultados durante explorações anteriores para desempatar e reportar modelos. Por isso, eles são mantidos como **holdout exploratório**, não como estimativa independente e não viesada de desempenho final. A independência não pode ser restaurada retroativamente.

A partir desta etapa, as novas comparações seguem um protocolo fixado antes de calcular seus resultados:

1. previsão em kg usa correção de retransfomação; o fator global é a referência e o fator por cluster é sensibilidade;
2. candidatos: OLS com elasticidade única, OLS com elasticidade por cluster e Huber com elasticidade única;
3. seleção prospectiva: expanding window a partir de 55 observações, prevendo a próxima observação registrada;
4. métricas primárias: WMAPE e viés agregado; RMSE é secundária para erros grandes e MAPE é diagnóstico de dias pequenos;
5. estabilidade retrospectiva: LOO e bootstrap estratificado por cluster.

Esse protocolo não transforma o holdout já visto em teste final. Para isso, seriam necessários novos dados futuros.

## 8. Retransformação: correção de smearing e seu risco de incerteza

Como a regressão é ajustada em `ln(volume)`, aplicar apenas a exponencial estima uma quantidade próxima da mediana condicional, não necessariamente o volume médio esperado. O fator de Duan, `mean(exp(resíduo))`, corrige essa retransfomação sem assumir normalidade.

O fator global é a referência por ser estimado com 76 resíduos. Como a escala de volume difere entre clusters, também calculamos fator por cluster como sensibilidade. Ele não é adotado automaticamente: no FDS, dez observações tornam esse fator potencialmente instável. A incerteza de ambos entra no bootstrap da etapa 10.

In [12]:
def ajustar_candidato(df, metodo="OLS", interacoes=False):
    base, X = preparar(df, interacoes=interacoes)
    if metodo == "OLS":
        modelo = sm.OLS(base["ln_volume"], X).fit()
    elif metodo == "Huber":
        modelo = sm.RLM(base["ln_volume"], X, M=sm.robust.norms.HuberT()).fit()
    else:
        raise ValueError(f"Método não reconhecido: {metodo}")
    return modelo, base


def fatores_smearing(modelo, base, por_cluster=False):
    exp_residuo = np.exp(np.asarray(modelo.resid, dtype=float))
    fator_global = float(exp_residuo.mean())
    if not por_cluster:
        return fator_global
    tabela = pd.DataFrame({"Cluster": base["Cluster"].to_numpy(), "exp_residuo": exp_residuo})
    fatores = tabela.groupby("Cluster", observed=True)["exp_residuo"].mean().to_dict()
    return fatores, fator_global


def prever_em_kg(modelo, df, interacoes=False, correcao="global", fatores=None):
    base, X = preparar(df, interacoes=interacoes, colunas_modelo=modelo.params.index)
    previsao = np.exp(np.asarray(modelo.predict(X), dtype=float))
    if correcao == "nenhuma":
        multiplicador = 1.0
    elif correcao == "global":
        multiplicador = fatores
    elif correcao == "cluster":
        fatores_cluster, fator_global = fatores
        multiplicador = base["Cluster"].map(fatores_cluster).fillna(fator_global).to_numpy()
    else:
        raise ValueError(f"Correção não reconhecida: {correcao}")
    return np.asarray(previsao * multiplicador, dtype=float)


def resumir_metricas(real, previsto):
    real = np.asarray(real, dtype=float)
    previsto = np.asarray(previsto, dtype=float)
    erro = real - previsto
    return {
        "RMSE (kg)": np.sqrt(np.mean(erro**2)),
        "WMAPE volume (%)": np.abs(erro).sum() / real.sum() * 100,
        "MAPE incidência (%)": (np.abs(erro) / real).mean() * 100,
        "Viés agregado (%)": (previsto.sum() - real.sum()) / real.sum() * 100,
    }


# Referência no ajuste completo; os fatores de cada fold são sempre recalculados no próprio fold.
modelo_ols_completo, base_ols_completo = ajustar_candidato(treino, "OLS", interacoes=False)
fator_global_completo = fatores_smearing(modelo_ols_completo, base_ols_completo)
fatores_cluster_completo = fatores_smearing(modelo_ols_completo, base_ols_completo, por_cluster=True)

fatores_smearing_tabela = pd.DataFrame({
    "Especificação": ["Global"] + [f"Por cluster: {ROTULO_CLUSTER[c]}" for c in ORDEM_CLUSTER],
    "Fator de smearing": [fator_global_completo] + [fatores_cluster_completo[0].get(c, np.nan) for c in ORDEM_CLUSTER],
})
display(fatores_smearing_tabela.round(3))

,Especificação,Fator de smearing
0,Global,1.118
1,Por cluster: Segunda,1.020
2,Por cluster: Terça + Quarta + Quinta,1.098
3,Por cluster: Sexta,1.112
4,Por cluster: Sábado + Domingo,1.331


**Resultado da correção.** O fator global é aproximadamente 1,12. Na validação temporal, ele reduz o viés agregado do OLS de −28,14% para −20,40% e reduz WMAPE de 41,51% para 38,16%. O MAPE aumenta porque elevar previsões penaliza relativamente mais dias de volume muito baixo; como WMAPE e viés são os critérios primários, o fator global é mantido.

O fator por cluster não melhora WMAPE nem viés na mesma validação e o fator do FDS tem intervalo bootstrap muito largo. Portanto, ele é evidência de heterogeneidade possível, não uma calibração suficientemente estável para substituir o fator global.

## 9. Validação temporal interna: desempenho prospectivo sem consultar o holdout

A cada data de corte, o modelo é ajustado apenas em observações anteriores e prevê a próxima observação registrada. A primeira previsão ocorre após 55 observações — limiar próximo ao ponto em que a elasticidade passa a mostrar sinal estável na análise anterior. Assim, são produzidas previsões one-step-ahead para as 21 observações restantes do treino.

Esse procedimento ainda tem pouca informação sobre FDS e não substitui dados futuros; sua função é comparar candidatos sem reutilizar os 10 dias externos já vistos.

In [13]:
ORDENADO = treino.sort_values("Data").reset_index(drop=True)
CANDIDATOS_FIXADOS = {
    "OLS | elasticidade única": ("OLS", False),
    "OLS | elasticidade por cluster": ("OLS", True),
    "Huber | elasticidade única": ("Huber", False),
}


def previsoes_expanding(df, metodo, interacoes, inicio_treino=55, correcao="global"):
    linhas = []
    for corte in range(inicio_treino, len(df)):
        treino_fold = df.iloc[:corte].copy()
        proximo = df.iloc[[corte]].copy()
        modelo, base_fold = ajustar_candidato(treino_fold, metodo, interacoes)
        if correcao == "global":
            fatores = fatores_smearing(modelo, base_fold)
        elif correcao == "cluster":
            fatores = fatores_smearing(modelo, base_fold, por_cluster=True)
        else:
            fatores = None
        previsto = prever_em_kg(modelo, proximo, interacoes, correcao, fatores)[0]
        linhas.append({
            "Data": proximo["Data"].iloc[0],
            "Dia da Semana": proximo["Dia da Semana"].iloc[0],
            "Real": proximo["Volume Realizado (kg)"].iloc[0],
            "Previsto": previsto,
        })
    return pd.DataFrame(linhas)


previsoes_cv = {}
linhas_cv = []
for nome, (metodo, interacoes) in CANDIDATOS_FIXADOS.items():
    previsoes = previsoes_expanding(ORDENADO, metodo, interacoes, correcao="global")
    previsoes_cv[nome] = previsoes
    linhas_cv.append({"Candidato": nome, "N previsões": len(previsoes), **resumir_metricas(previsoes["Real"], previsoes["Previsto"])})

tabela_cv = pd.DataFrame(linhas_cv).sort_values(["WMAPE volume (%)", "RMSE (kg)"])
display(tabela_cv.round(2))

# Avalia somente a calibração da retransfomação no candidato OLS compartilhado.
calibracoes = []
for correcao in ("nenhuma", "global", "cluster"):
    previsoes = previsoes_expanding(ORDENADO, "OLS", False, correcao=correcao)
    calibracoes.append({"Correção": correcao.capitalize(), "N previsões": len(previsoes), **resumir_metricas(previsoes["Real"], previsoes["Previsto"])})
tabela_calibracao = pd.DataFrame(calibracoes)
display(tabela_calibracao.round(2))

,Candidato,N previsões,RMSE (kg),WMAPE volume (%),MAPE incidência (%),Viés agregado (%)
2,Huber | elasticidade única,21,13.83,32.70,91.37,-16.38
1,OLS | elasticidade por cluster,21,14.48,35.40,99.16,-18.01
0,OLS | elasticidade única,21,16.14,38.16,93.61,-20.40


,Correção,N previsões,RMSE (kg),WMAPE volume (%),MAPE incidência (%),Viés agregado (%)
0,Nenhuma,21,17.55,41.51,87.34,-28.14
1,Global,21,16.14,38.16,93.61,-20.40
2,Cluster,21,16.39,38.53,98.81,-22.91


**Resultado da validação temporal e escolha metodológica.** Huber com elasticidade única obtém o menor WMAPE (32,70%), menor RMSE (13,83 kg) e menor viés agregado em magnitude (−16,38%) entre os candidatos fixados. Ele também melhora MAPE frente ao OLS de elasticidade única. Assim, sob o protocolo prospectivo definido, o candidato selecionado é **Huber com quatro interceptos e elasticidade compartilhada**. OLS permanece como análise de sensibilidade simples e transparente, não como modelo operacional principal.

Essa comparação responde a uma pergunta prospectiva: dado apenas o passado disponível, qual candidato prevê melhor a próxima observação? Ela é diferente de LOO e bootstrap, que medem a estabilidade retrospectiva do ajuste final com os 76 dias completos. As duas famílias de reamostragem são complementares, não redundantes.

## 10. Estabilidade retrospectiva: LOO e bootstrap

LOO mede quanto a elasticidade muda ao retirar cada dia: é um diagnóstico de influência, não uma regra de exclusão. O bootstrap reamostra dentro dos clusters para preservar seus tamanhos relativos e estima intervalos para β, para a diferença OLS–Huber, para a sensibilidade ao FDS e para os fatores de smearing.

O bootstrap é estratificado por cluster, não temporal em blocos. A opção é deliberada: com apenas 76 dias, blocos temporais exigiriam um tamanho de bloco arbitrário e deixariam poucos blocos para reamostrar. Esse bootstrap quantifica estabilidade amostral, não dependência serial.

In [14]:
# LOO: influência de cada dia na elasticidade OLS e Huber compartilhadas.
linhas_loo = []
for indice in range(len(treino)):
    amostra = treino.drop(index=treino.index[indice])
    modelo_loo_ols, _ = ajustar_candidato(amostra, "OLS", interacoes=False)
    modelo_loo_huber, _ = ajustar_candidato(amostra, "Huber", interacoes=False)
    linhas_loo.append({
        "Data removida": treino.iloc[indice]["Data"],
        "Dia da Semana": treino.iloc[indice]["Dia da Semana"],
        "β OLS sem dia": modelo_loo_ols.params["ln_preco"],
        "β Huber sem dia": modelo_loo_huber.params["ln_preco"],
    })
loo = pd.DataFrame(linhas_loo)
loo["Δβ OLS"] = loo["β OLS sem dia"] - modelo_ols_completo.params["ln_preco"]
modelo_huber_completo, base_huber_completo = ajustar_candidato(treino, "Huber", interacoes=False)
loo["Δβ Huber"] = loo["β Huber sem dia"] - modelo_huber_completo.params["ln_preco"]

resumo_loo = pd.DataFrame({
    "Método": ["OLS", "Huber"],
    "β completo": [modelo_ols_completo.params["ln_preco"], modelo_huber_completo.params["ln_preco"]],
    "Menor β LOO": [loo["β OLS sem dia"].min(), loo["β Huber sem dia"].min()],
    "Maior β LOO": [loo["β OLS sem dia"].max(), loo["β Huber sem dia"].max()],
    "Maior |Δβ|": [loo["Δβ OLS"].abs().max(), loo["Δβ Huber"].abs().max()],
})
display(resumo_loo.round(3))
loo_mais_influentes = loo.reindex(loo["Δβ OLS"].abs().sort_values(ascending=False).head(8).index).copy()
loo_mais_influentes[["β OLS sem dia", "β Huber sem dia", "Δβ OLS", "Δβ Huber"]] = loo_mais_influentes[["β OLS sem dia", "β Huber sem dia", "Δβ OLS", "Δβ Huber"]].round(3)
display(loo_mais_influentes)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(loo["Data removida"], loo["β OLS sem dia"], marker="o", ms=3, label="OLS")
ax.plot(loo["Data removida"], loo["β Huber sem dia"], marker="o", ms=3, label="Huber")
ax.axhline(modelo_ols_completo.params["ln_preco"], color="#1f77b4", ls="--", lw=1)
ax.axhline(modelo_huber_completo.params["ln_preco"], color="#ff7f0e", ls="--", lw=1)
ax.set(title="LOO: elasticidade após retirar cada dia", xlabel="Data retirada", ylabel="Elasticidade β")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_ols_loo_elasticidade.png", dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)

# Bootstrap estratificado: preserva a quantidade observada em cada cluster.
rng = np.random.default_rng(42)
base_boot = treino.assign(Cluster=treino["Dia da Semana"].map(definir_cluster)).reset_index(drop=True)
indices_cluster = {cluster: base_boot.index[base_boot["Cluster"] == cluster].to_numpy() for cluster in ORDEM_CLUSTER}
registros_boot = []
for _ in range(1000):
    indices = np.concatenate([rng.choice(indices_cluster[c], size=len(indices_cluster[c]), replace=True) for c in ORDEM_CLUSTER])
    amostra = base_boot.iloc[indices].drop(columns="Cluster").reset_index(drop=True)
    ols_boot, base_ols_boot = ajustar_candidato(amostra, "OLS", interacoes=False)
    huber_boot, _ = ajustar_candidato(amostra, "Huber", interacoes=False)
    fatores_cluster_boot = fatores_smearing(ols_boot, base_ols_boot, por_cluster=True)
    amostra_sem_fds = amostra[~amostra["Dia da Semana"].isin(["Sábado", "Domingo"])].reset_index(drop=True)
    ols_sem_fds, _ = ajustar_candidato(amostra_sem_fds, "OLS", interacoes=False)
    registros_boot.append({
        "β OLS": ols_boot.params["ln_preco"],
        "β Huber": huber_boot.params["ln_preco"],
        "Δ OLS−Huber": ols_boot.params["ln_preco"] - huber_boot.params["ln_preco"],
        "β OLS sem FDS": ols_sem_fds.params["ln_preco"],
        "Δ β sem−com FDS": ols_sem_fds.params["ln_preco"] - ols_boot.params["ln_preco"],
        "Smearing global": fatores_cluster_boot[1],
        **{f"Smearing {c}": fatores_cluster_boot[0].get(c, np.nan) for c in ORDEM_CLUSTER},
    })

bootstrap = pd.DataFrame(registros_boot)
resumo_bootstrap = pd.DataFrame([
    {"Quantidade": coluna, "Média": bootstrap[coluna].mean(), "IC 2,5%": bootstrap[coluna].quantile(0.025), "IC 97,5%": bootstrap[coluna].quantile(0.975)}
    for coluna in bootstrap.columns
])
display(resumo_bootstrap.round(3))

fator_smearing_huber = fatores_smearing(modelo_huber_completo, base_huber_completo)
modelo_operacional = pd.DataFrame([{
    "Modelo selecionado": "Huber: 4 interceptos + elasticidade única",
    "Elasticidade β": modelo_huber_completo.params["ln_preco"],
    "Fator global de smearing": fator_smearing_huber,
    "Clusters de nível": "Segunda / TerQuaQui / Sexta / FDS",
}])
display(modelo_operacional.round(3))

,Método,β completo,Menor β LOO,Maior β LOO,Maior |Δβ|
0,OLS,-12.458,-14.200,-11.343,1.742
1,Huber,-14.099,-14.874,-13.600,0.775


,Data removida,Dia da Semana,β OLS sem dia,β Huber sem dia,Δβ OLS,Δβ Huber
57,2025-10-11,Sábado,-14.200,-14.874,-1.742,-0.775
68,2025-10-23,Quinta,-11.343,-13.681,1.115,0.418
60,2025-10-15,Quarta,-11.751,-13.630,0.707,0.469
64,2025-10-19,Domingo,-12.960,-14.493,-0.502,-0.394
22,2025-08-30,Sábado,-12.888,-14.292,-0.429,-0.193
69,2025-10-24,Sexta,-12.035,-13.600,0.423,0.499
3,2025-08-06,Quarta,-12.867,-14.394,-0.409,-0.294
73,2025-10-29,Quarta,-12.866,-14.656,-0.408,-0.557


<Figure size 1000x450 with 1 Axes>

,Quantidade,Média,"IC 2,5%","IC 97,5%"
0,β OLS,-12.497,-17.075,-7.803
1,β Huber,-13.746,-17.416,-9.297
2,Δ OLS−Huber,1.249,-0.768,3.355
3,β OLS sem FDS,-14.790,-18.548,-10.751
4,Δ β sem−com FDS,-2.294,-5.204,0.388
5,Smearing global,1.110,1.065,1.159
6,Smearing Segunda,1.020,1.007,1.035
7,Smearing TerQuaQui,1.094,1.051,1.144
8,Smearing Sexta,1.104,1.051,1.160
9,Smearing FimDeSemana,1.295,1.060,1.568


,Modelo selecionado,Elasticidade β,Fator global de smearing,Clusters de nível
0,Huber: 4 interceptos + elasticidade única,-14.099,1.061,Segunda / TerQuaQui / Sexta / FDS


**Leitura do LOO e do bootstrap.** OLS completo estima β = −12,46. Ao retirar somente 11/10 (sábado), β passa a −14,20: deslocamento de −1,74, cerca de 14% do módulo de β. Ao retirar 23/10 (quinta), β passa a −11,34: deslocamento de +1,12, cerca de 9%. Esses dois dias puxam a inclinação OLS em direções opostas; eles não devem ser removidos, mas explicam por que OLS é mais frágil à composição da amostra.

Huber completo estima β = −14,10. No LOO, ele varia de −14,87 a −13,60, com maior deslocamento de 0,78 (cerca de 5,5%). Portanto, a vantagem de Huber não é apenas o WMAPE temporal: sua elasticidade é menos dependente de um único dia divergente da curva.

O bootstrap mantém a incerteza honesta: os intervalos de β são largos e o intervalo da diferença OLS−Huber contém zero. Isso significa que a amostra não identifica com precisão uma “elasticidade verdadeira” distinta por método. A escolha por Huber vem do desempenho prospectivo e da menor influência LOO, não de um teste de diferença entre métodos. A diferença de β com e sem FDS também tem intervalo que inclui zero; FDS permanece no modelo pelo forte efeito de nível, enquanto sua contribuição à inclinação segue sendo análise de sensibilidade.

## Decisão final e uso do modelo

**Modelo operacional escolhido:** regressão robusta de Huber em `ln(volume)`, com quatro interceptos de nível — Segunda; Terça+Quarta+Quinta; Sexta; e FDS — e uma elasticidade compartilhada. No ajuste completo, a elasticidade é aproximadamente **β = −14,10**. A previsão em kg aplica o fator global de smearing estimado no treino do ajuste correspondente.

A escolha é decidida, não apenas sugerida: Huber vence os três critérios prospectivos relevantes na validação temporal interna (WMAPE, RMSE e viés agregado) e sua elasticidade é menos influenciável no LOO. OLS com β = −12,46 permanece no relatório como sensibilidade de simplicidade e para mostrar o custo de dar peso integral a todos os resíduos, mas não é o modelo operacional preferido após essas evidências.

A estrutura de calendário também fica decidida: mês, quinzena e semana do mês não entram; o dia da semana entra; Terça+Quarta+Quinta formam o grupo parcimonioso; FDS mantém intercepto próprio pelo BIC e teste F, embora sua elasticidade própria seja rejeitada e sua previsão fora da amostra não tenha sido observada.

**Limites que acompanham a decisão:** o holdout original de 10 dias já foi reutilizado e é apenas exploratório; o intervalo de β é amplo; e FDS não possui teste externo. Isso não impede a escolha atual, mas impede afirmar precisão final não viesada sem novos dados futuros. A otimização deve respeitar o suporte de preço observado por cluster e incluir FDS somente quando sábado/domingo fizerem parte do horizonte.